# Mô hình ngôn ngữ nhỏ cho câu hỏi pháp luật Việt Nam
### Huấn luyện trên dữ liệu gốc + dữ liệu mở rộng từ Claude Haiku 4.5

**Yêu cầu:**
- Kaggle Notebook
- Thư mục `data/` với dữ liệu gốc và dữ liệu sinh
- Set `ANTHROPIC_API_KEY` trong Add-ons -> Secrets (dùng cho khâu chấm điểm tam đoạn luận)

**Cấu trúc thư mục data:**
```
data/
├── original/              (440 mẫu gốc — dùng để đánh giá)
│   ├── mc.jsonl
│   ├── nli.jsonl
│   └── syllo.jsonl
└── generated/             (dữ liệu sinh - dùng để train)
    ├── train_mc.jsonl
    ├── train_nli.jsonl
    ├── train_syllogism.jsonl
```

**Các bước thực hiện:**
1. Upload dữ liệu
2. Giai đoạn 1: huấn luyện 3 adapter QLoRA cho ba tác vụ
3. Hợp nhất 3 adapter bằng nội suy tuyến tính
4. Giai đoạn 2: tinh chỉnh toàn phần (Full fine tunning)
5. Đánh giá (trắc nghiệm và NLI chấm theo đáp án, tam đoạn luận dùng Claude Haiku 4.5)

---
## 0. Cài đặt thư viện

In [2]:
# Cài đặt thư viện
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.12.0" peft accelerate bitsandbytes
!pip install -q "transformers>=4.45.0" "datasets>=2.18.0"
!pip install -q anthropic safetensors

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 64.3 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 60.4 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 1. Khởi tạo và cấu hình

In [1]:
# Import thư viện
import os
import json
import random
import re
import time
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

# Check GPU
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Vào Runtime > Change runtime type > chọn T4 GPU")

GPU_NAME = torch.cuda.get_device_name(0)
GPU = "A100" if "A100" in GPU_NAME else "T4"
print(f"Đang dùng GPU: {GPU_NAME} (cấu hình: {GPU})")

Đang dùng GPU: Tesla T4 (cấu hình: T4)


In [3]:
# Cấu hình

# Tham số phụ thuộc GPU
BATCH = 8 if GPU == "A100" else 2
ACCUM = 4 if GPU == "A100" else 16
BF16 = GPU == "A100"
FP16 = not BF16

# Mô hình nền
BASE_MODEL = "VLSP2025-LegalSML/qwen3-4b-legal-pretrain"
MAX_SEQ_LENGTH = 2048

# Cấu hình LoRA
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Huấn luyện hai giai đoạn
STAGE1_LR = 2e-4
STAGE1_EPOCHS = 3
STAGE2_LR = 5e-5
STAGE2_EPOCHS = 2

# Set SEED để cố định kết quả random
SEED = 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

SYSTEM_PROMPT = "Bạn là một chuyên gia pháp luật Việt Nam."

print(f"Batch hiệu dụng: {BATCH}x{ACCUM} = {BATCH*ACCUM}, độ chính xác: {'bf16' if BF16 else 'fp16'}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}")

Batch hiệu dụng: 2x16 = 32, độ chính xác: fp16
LoRA: rank=16, alpha=32


---
## 2. Đọc dữ liệu

In [47]:
def find_data_dir():
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "original" in dirs and "generated" in dirs:
            return root

DATA_DIR = find_data_dir()
ORIG_DIR = f"{DATA_DIR}/original"
GEN_DIR  = f"{DATA_DIR}/generated"
ADAPTER_DIR   = "/kaggle/working/adapters"
MODEL_OUT_DIR = "/kaggle/working/model_final"
RESULTS_DIR   = "/kaggle/working/results"

for d in [ADAPTER_DIR, MODEL_OUT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"GEN_DIR:       generated/ - {len(os.listdir(GEN_DIR))} files")
print(f"ORIG_DIR:      original/  - {len(os.listdir(ORIG_DIR))} files")
print(f"ADAPTER_DIR:   {ADAPTER_DIR}")
print(f"MODEL_OUT_DIR: {MODEL_OUT_DIR}")
print(f"RESULTS_DIR:   {RESULTS_DIR}")

GEN_DIR:       generated/ - 3 files
ORIG_DIR:      original/  - 3 files
ADAPTER_DIR:   /kaggle/working/adapters
MODEL_OUT_DIR: /kaggle/working/model_final
RESULTS_DIR:   /kaggle/working/results


In [49]:
# TĐọc dữ liệu huấn luyện 

def load_jsonl(path):
    if not os.path.exists(path):
        print(f"Không tìm thấy: {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_mc = load_jsonl(f"{GEN_DIR}/train_mc.jsonl")
train_nli = load_jsonl(f"{GEN_DIR}/train_nli.jsonl")
train_syllo = load_jsonl(f"{GEN_DIR}/train_syllogism.jsonl")

# Tập tổng hợp cho giai đoạn 2 được gộp trực tiếp từ 3 file
train_combined = train_mc + train_nli + train_syllo
random.shuffle(train_combined)

print("Dữ liệu huấn luyện:")
print(f"  Trắc nghiệm: {len(train_mc)} mẫu")
print(f"  NLI: {len(train_nli)} mẫu")
print(f"  Tam đoạn luận: {len(train_syllo)} mẫu")
print(f"  Tổng hợp (gộp 3 tác vụ): {len(train_combined)} mẫu")

if not train_combined:
    print("\nChưa có dữ liệu. Hãy chạy lại data_generation.py và upload thư mục data")

Dữ liệu huấn luyện:
  Trắc nghiệm: 894 mẫu
  NLI: 900 mẫu
  Tam đoạn luận: 552 mẫu
  Tổng hợp (gộp 3 tác vụ): 2346 mẫu


In [23]:
# Dữ liệu gốc (440 mẫu) để đánh giá
eval_mc = load_jsonl(f"{ORIG_DIR}/mc.jsonl")
eval_nli = load_jsonl(f"{ORIG_DIR}/nli.jsonl")
eval_syllo = load_jsonl(f"{ORIG_DIR}/syllo.jsonl")

print("Dữ liệu đánh giá (gốc 440 mẫu):")
print(f"  Trắc nghiệm: {len(eval_mc)} | NLI: {len(eval_nli)} | Tam đoạn luận: {len(eval_syllo)}")

# Hai tác vụ MC và NLI cần trường 'answer' (số nguyên) để chấm điểm
if eval_mc and "answer" not in eval_mc[0]:
    print("Cảnh báo: dữ liệu trắc nghiệm thiếu trường 'answer', kiểm tra lại data/original/mc.jsonl")

Dữ liệu đánh giá (gốc 440 mẫu):
  Trắc nghiệm: 146 | NLI: 150 | Tam đoạn luận: 144


---
## 3. Giai đoạn 1 - Huấn luyện QLoRA theo từng tác vụ

In [25]:
# Tải mô hình base và gắn adapter LoRA
from unsloth import FastLanguageModel

def load_base_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    return model, tokenizer

def attach_lora(model):
    return FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

print("Đã chuẩn bị xong các hàm nạp mô hình")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Đã chuẩn bị xong các hàm nạp mô hình


In [26]:
# Hàm huấn luyện một adapter cho một tác vụ
from unsloth import UnslothTrainer, UnslothTrainingArguments
from datasets import Dataset

def train_adapter(task, data):
    print(f"\nBắt đầu huấn luyện tác vụ '{task}' với {len(data)} mẫu")

    if not data:
        print(f"Bỏ qua '{task}' vì không có dữ liệu")
        return None

    model, tokenizer = load_base_model()
    model = attach_lora(model)

    def to_text(ex):
        return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

    # Chỉ giữ trường 'messages' để tránh lỗi Arrow do trường 'answer' lẫn lộn kiểu dữ liệu
    ds = Dataset.from_list([{"messages": ex["messages"]} for ex in data]).map(to_text, remove_columns=["messages"])

    out_dir = f"{ADAPTER_DIR}/{task}"

    trainer = UnslothTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        args=UnslothTrainingArguments(
            output_dir=out_dir,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=ACCUM,
            warmup_ratio=0.05,
            num_train_epochs=STAGE1_EPOCHS,
            learning_rate=STAGE1_LR,
            bf16=BF16,
            fp16=FP16,
            logging_steps=5,
            optim="adamw_8bit",
            seed=SEED,
            report_to="none",
        ),
    )
    trainer.train()

    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f"Đã lưu adapter '{task}' tại {out_dir}")

    del model, trainer
    torch.cuda.empty_cache()
    return out_dir

print("Đã chuẩn bị xong hàm huấn luyện")

Đã chuẩn bị xong hàm huấn luyện


In [27]:
# Huấn luyện ba adapter cho ba tác vụ
adapter_paths = {}

for task, data in [("mc", train_mc), ("nli", train_nli), ("syllogism", train_syllo)]:
    path = train_adapter(task, data)
    if path:
        adapter_paths[task] = path

print(f"\nĐã huấn luyện {len(adapter_paths)} adapter: {list(adapter_paths.keys())}")

with open(f"{ADAPTER_DIR}/paths.json", "w") as f:
    json.dump(adapter_paths, f)


Bắt đầu huấn luyện tác vụ 'mc' với 894 mẫu
==((====))==  Unsloth 2026.5.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 552 | Num Epochs = 3 | Total steps = 54
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.599655
10,1.113764
15,0.962636
20,0.907755
25,0.890287
30,0.844030
35,0.849427
40,0.813223
45,0.791861
50,0.826138


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/adapters/syllogism/checkpoint-54/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/adapters/syllogism/tokenizer_config.json.


Đã lưu adapter 'syllogism' tại /kaggle/working/adapters/syllogism

Đã huấn luyện 3 adapter: ['mc', 'nli', 'syllogism']


In [28]:
# Hợp nhất ba adapter bằng nội suy tuyến tính
from safetensors.torch import load_file, save_file
import shutil

def merge_adapters(paths):
    tasks = list(paths.keys())
    print(f"Đang hợp nhất {len(tasks)} adapter: {tasks}")
    merged = None

    for t in tasks:
        adapter_file = f"{paths[t]}/adapter_model.safetensors"
        sd = load_file(adapter_file)
        weight = 1.0 / len(tasks)

        if merged is None:
            merged = {k: v.float() * weight for k, v in sd.items()}
        else:
            for k, v in sd.items():
                merged[k] += v.float() * weight

    merged_dir = f"{ADAPTER_DIR}/merged"
    os.makedirs(merged_dir, exist_ok=True)
    save_file({k: v.contiguous() for k, v in merged.items()}, f"{merged_dir}/adapter_model.safetensors")
    shutil.copy(f"{paths[tasks[0]]}/adapter_config.json", f"{merged_dir}/adapter_config.json")

    print(f"Đã hợp nhất, lưu tại {merged_dir}")
    return merged_dir

MERGED_ADAPTER = merge_adapters(adapter_paths)
print("Hoàn thành giai đoạn 1")

Đang hợp nhất 3 adapter: ['mc', 'nli', 'syllogism']
Đã hợp nhất, lưu tại /kaggle/working/adapters/merged
Hoàn thành giai đoạn 1


---
## 4. Giai đoạn 2 - Tinh chỉnh toàn phần (Full fine tunning)

In [29]:
# Nạp lại mô hình nền và gắn adapter đã hợp nhất
from peft import PeftModel

model, tokenizer = load_base_model()
model = PeftModel.from_pretrained(model, MERGED_ADAPTER, is_trainable=True)
print("Đã nạp adapter hợp nhất làm điểm khởi tạo cho giai đoạn 2")

==((====))==  Unsloth 2026.5.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Đã nạp adapter hợp nhất làm điểm khởi tạo cho giai đoạn 2


In [31]:
# Tinh chỉnh toàn phần trên toàn bộ dữ liệu tổng hợp
from datasets import Dataset

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

# Chỉ giữ trường 'messages' để tránh lỗi Arrow do trường 'answer' lẫn lộn kiểu dữ liệu (int vs null)
ds = Dataset.from_list([{"messages": ex["messages"]} for ex in train_combined]).map(to_text, remove_columns=["messages"])
print(f"Số mẫu huấn luyện giai đoạn 2: {len(ds)}")

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=UnslothTrainingArguments(
        output_dir=MODEL_OUT_DIR,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM,
        warmup_ratio=0.05,
        num_train_epochs=STAGE2_EPOCHS,
        learning_rate=STAGE2_LR,
        bf16=BF16,
        fp16=FP16,
        logging_steps=5,
        optim="adamw_8bit",
        seed=SEED,
        report_to="none",
    ),
)
trainer.train()

model.save_pretrained(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)
print(f"\nĐã lưu mô hình cuối cùng tại {MODEL_OUT_DIR}")

Map:   0%|          | 0/2346 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Số mẫu huấn luyện giai đoạn 2: 2346


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2346 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,346 | Num Epochs = 2 | Total steps = 148
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.685907
10,0.678084
15,0.680355
20,0.696156
25,0.673502
30,0.666488
35,0.652002
40,0.675915
45,0.672610
50,0.706334



Đã lưu mô hình cuối cùng tại /kaggle/working/model_final


---
## 5. Đánh giá

In [32]:
# Nạp mô hình ở chế độ suy luận
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_OUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Mô hình đã sẵn sàng để suy luận")

==((====))==  Unsloth 2026.5.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Mô hình đã sẵn sàng để suy luận


In [34]:
# Các hàm phục vụ đánh giá
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()  # Tắt cảnh báo lặp lại về max_new_tokens/max_length

def generate(messages, max_tokens=256):
    prompt_msgs = [m for m in messages if m["role"] in ("system", "user")]
    inputs = tokenizer.apply_chat_template(prompt_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

LETTER2IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

def parse_mc(text):
    m = re.search(r"[ABCD]", text.upper())
    return LETTER2IDX.get(m.group(0)) if m else None

def parse_nli(text):
    t = text.lower()
    if "không" in t:
        return 1
    if "có" in t:
        return 0
    return None

print("Đã chuẩn bị xong các hàm đánh giá")

Đã chuẩn bị xong các hàm đánh giá


In [35]:
# Đánh giá tác vụ trắc nghiệm và NLI bằng khớp chính xác
print("Đang đánh giá trắc nghiệm...")
mc_correct = 0
for i, ex in enumerate(eval_mc):
    pred = parse_mc(generate(ex["messages"], 8))
    if pred == ex["answer"]:
        mc_correct += 1
    if (i + 1) % 50 == 0:
        print(f"  đã chấm {i+1}/{len(eval_mc)}")

mc_acc = 100.0 * mc_correct / len(eval_mc)
print(f"Trắc nghiệm: {mc_acc:.2f}% ({mc_correct}/{len(eval_mc)})")

print("\nĐang đánh giá NLI...")
nli_correct = 0
for i, ex in enumerate(eval_nli):
    pred = parse_nli(generate(ex["messages"], 8))
    if pred == ex["answer"]:
        nli_correct += 1
    if (i + 1) % 50 == 0:
        print(f"  đã chấm {i+1}/{len(eval_nli)}")

nli_acc = 100.0 * nli_correct / len(eval_nli)
print(f"NLI: {nli_acc:.2f}% ({nli_correct}/{len(eval_nli)})")

Đang đánh giá trắc nghiệm...
  đã chấm 50/146
  đã chấm 100/146
Trắc nghiệm: 86.99% (127/146)

Đang đánh giá NLI...
  đã chấm 50/150
  đã chấm 100/150
  đã chấm 150/150
NLI: 89.33% (134/150)


In [42]:
# Đánh giá tam đoạn luận bằng mô hình giám khảo Claude
# Cần khai báo ANTHROPIC_API_KEY trong phần Add-ons -> Secrets
import anthropic
syllo_score = 50.0  # Giá trị mặc định nếu không có API key
JUDGE_MODEL = "claude-haiku-4-5-20251001"
GEN_TOKENS = 500      # Số token tối đa cho câu trả lời tam đoạn luận
N_EVAL = len(eval_syllo)
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    if api_key:
        client = anthropic.Anthropic(api_key=api_key)
        scores = []
        print(f"Đang chấm tam đoạn luận bằng Claude ({N_EVAL} mẫu)...", flush=True)
        for i, ex in enumerate(eval_syllo):
            t0 = time.time()
            answer = generate(ex["messages"], GEN_TOKENS)
            t_gen = time.time() - t0
            question = next(m["content"] for m in ex["messages"] if m["role"] == "user")
            score_str = "?"
            try:
                resp = client.messages.create(
                    model=JUDGE_MODEL,
                    max_tokens=10,
                    messages=[{
                        "role": "user",
                        "content": (
                            "Bạn là giám khảo chấm điểm câu trả lời tam đoạn luận pháp lý. "
                            "Cho điểm từ 0.0 đến 1.0 dựa trên tính đúng đắn của tiền đề lớn, tiền đề nhỏ "
                            "và tính hợp lý của kết luận. Chỉ trả về một con số, không giải thích.\n\n"
                            f"Câu hỏi:\n{question}\n\nCâu trả lời:\n{answer}"
                        )
                    }],
                )
                text = resp.content[0].text
                m = re.search(r"[01](?:\.\d+)?", text)
                if m:
                    s = min(1.0, float(m.group(0)))
                    scores.append(s)
                    score_str = f"{s:.2f}"
            except Exception as e:
                score_str = f"lỗi ({type(e).__name__})"
            print(f"  mẫu {i+1}/{N_EVAL}: sinh {t_gen:.1f}s, điểm {score_str}", flush=True)
        if scores:
            syllo_score = 100.0 * sum(scores) / len(scores)
        print(f"\nTam đoạn luận: {syllo_score:.2f}% (đã chấm {len(scores)}/{N_EVAL} mẫu)")
    else:
        print("Chưa có ANTHROPIC_API_KEY, dùng điểm mặc định 50%")
except Exception as e:
    print(f"Bỏ qua đánh giá tam đoạn luận: {e}")
print(f"\nTam đoạn luận: {syllo_score:.2f}%")

Đang chấm tam đoạn luận bằng Claude (144 mẫu)...
  mẫu 1/144: sinh 29.1s, điểm 0.30
  mẫu 2/144: sinh 23.8s, điểm 0.60
  mẫu 3/144: sinh 27.7s, điểm 0.60
  mẫu 4/144: sinh 26.3s, điểm 0.60
  mẫu 5/144: sinh 37.2s, điểm 0.40
  mẫu 6/144: sinh 26.3s, điểm 0.70
  mẫu 7/144: sinh 20.0s, điểm 0.60
  mẫu 8/144: sinh 23.1s, điểm 0.70
  mẫu 9/144: sinh 35.6s, điểm 0.60
  mẫu 10/144: sinh 29.2s, điểm 0.40
  mẫu 11/144: sinh 29.4s, điểm 0.60
  mẫu 12/144: sinh 29.7s, điểm 0.30
  mẫu 13/144: sinh 23.5s, điểm 0.40
  mẫu 14/144: sinh 30.7s, điểm 0.60
  mẫu 15/144: sinh 32.0s, điểm 0.60
  mẫu 16/144: sinh 37.3s, điểm 0.40
  mẫu 17/144: sinh 35.3s, điểm 0.60
  mẫu 18/144: sinh 28.6s, điểm 0.60
  mẫu 19/144: sinh 33.6s, điểm 0.60
  mẫu 20/144: sinh 18.7s, điểm 0.30
  mẫu 21/144: sinh 28.1s, điểm 0.20
  mẫu 22/144: sinh 36.9s, điểm 0.60
  mẫu 23/144: sinh 25.8s, điểm 0.80
  mẫu 24/144: sinh 36.8s, điểm 0.30
  mẫu 25/144: sinh 30.2s, điểm 0.30
  mẫu 26/144: sinh 19.6s, điểm 0.20
  mẫu 27/144: sinh 26.8s

In [45]:
# Tổng hợp kết quả cuối cùng
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

avg = (mc_acc + nli_acc + syllo_score) / 3
results = {
    "mc": round(mc_acc, 2),
    "nli": round(nli_acc, 2),
    "syllo": round(syllo_score, 2),
    "avg": round(avg, 2),
    "config": {
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "stage1_lr": STAGE1_LR,
        "stage1_epochs": STAGE1_EPOCHS,
        "stage2_lr": STAGE2_LR,
        "stage2_epochs": STAGE2_EPOCHS,
    },
}
with open(f"{RESULTS_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Kết quả cuối cùng")
print(f"  Trắc nghiệm:   {results['mc']:.2f}%")
print(f"  NLI:           {results['nli']:.2f}%")
print(f"  Tam đoạn luận: {results['syllo']:.2f}%")
print(f"  Trung bình:    {results['avg']:.2f}%")
print("\nĐối chiếu tham khảo:")
print(f"  Bosch@AI (hạng 1): 81.08%")
print(f"  MinLegal (hạng 2): 85.24%")
print(f"  Mô hình của nhóm:    {results['avg']:.2f}%")

Kết quả cuối cùng
  Trắc nghiệm:   86.99%
  NLI:           89.33%
  Tam đoạn luận: 56.70%
  Trung bình:    77.67%

Đối chiếu tham khảo:
  Bosch@AI (hạng 1): 81.08%
  MinLegal (hạng 2): 85.24%
  Mô hình của nhóm:    77.67%


In [46]:
## Hoàn tất
print("Các file đã được lưu sẵn tại:")
print(f"  Adapters : {ADAPTER_DIR}")
print(f"  Model    : {MODEL_OUT_DIR}")
print(f"  Results  : {RESULTS_DIR}")
print("\nVào tab Output bên phải để tải xuống.")

Các file đã được lưu sẵn tại:
  Adapters : /kaggle/working/adapters
  Model    : /kaggle/working/model_final
  Results  : /kaggle/working/results

Vào tab Output bên phải để tải xuống.
